In [42]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [43]:
df = pd.read_csv("/content/qoute_dataset.csv")

In [44]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [45]:
df['quote'][0]

'“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”'

In [46]:
df.shape

(3038, 2)

In [47]:
df.isnull().sum()

,0
quote,0
Author,0


In [48]:
quote = df['quote']
quote.head()

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


In [49]:
quote = quote.str.lower()

In [50]:
import string
translator = str.maketrans('','', string.punctuation)
quote = quote.apply(lambda x: x.translate(translator))

In [51]:
quote.head()

,quote
0,“the world as we have created it is a process ...
1,“it is our choices harry that show what we tru...
2,“there are only two ways to live your life one...
3,“the person be it gentleman or lady who has no...
4,“imperfection is beauty madness is genius and ...


In [52]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [53]:
vocab_size = 10000

tokenizer = Tokenizer(num_words=vocab_size,  oov_token="<OOV>")
tokenizer.fit_on_texts(quote)

In [54]:
word_index = tokenizer.word_index
print(len(word_index))
list(word_index.items())[:10]

8979


[('<OOV>', 1),
 ('the', 2),
 ('you', 3),
 ('to', 4),
 ('and', 5),
 ('a', 6),
 ('i', 7),
 ('is', 8),
 ('of', 9),
 ('that', 10)]

In [55]:
sequence = tokenizer.texts_to_sequences(quote)

In [56]:
for i in range(3):
  print(quote[i])

“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
“it is our choices harry that show what we truly are far more than our abilities”
“there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”


In [57]:
for i in range(3):
  print(sequence[i])

[714, 63, 30, 20, 17, 947, 11, 8, 6, 1157, 9, 71, 294, 11, 146, 13, 810, 105, 753, 71, 2462]
[948, 8, 71, 872, 374, 10, 434, 22, 20, 466, 15, 295, 53, 55, 71, 3677]
[1338, 15, 54, 202, 715, 4, 82, 16, 37, 38, 8, 30, 330, 94, 8, 6, 1158, 2, 102, 8, 30, 330, 127, 8, 6, 3678]


In [58]:
X = []
y = []

for seq in sequence:
  for i in range(1,len(seq)):
    input_seq = seq[:i]
    output_seq = seq[i]
    X.append(input_seq)
    y.append(output_seq)

In [59]:
len(X)

85271

In [60]:
len(y)

85271

In [61]:
max_len = max(len(x) for x in X)
print(max_len)

745


In [62]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_padded = pad_sequences(X,maxlen=max_len,padding='pre')

In [63]:
y = np.array(y)

In [64]:
X_padded.shape

(85271, 745)

In [65]:
from tensorflow.keras.utils import to_categorical
y_one_hot = to_categorical(y,num_classes=vocab_size)

In [66]:
y.shape

(85271,)

In [67]:
y_one_hot.shape

(85271, 10000)

In [68]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, SimpleRNN, Dropout

In [69]:
embedding_dim = 50
rnn_units = 128

In [70]:
rnn_model = Sequential()

rnn_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)
)
rnn_model.add(SimpleRNN(units=rnn_units))
rnn_model.add(Dense(units=vocab_size, activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [71]:
rnn_model.compile(
    optimizer = 'adam',
    loss = 'categorical_crossentropy',
    metrics = ['accuracy']
)

In [72]:
rnn_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [73]:
lstm_model = Sequential()
lstm_model.add(
    Embedding(input_dim=vocab_size, output_dim=128, input_length=max_len)
)

lstm_model.add(LSTM(256, return_sequences=True))
lstm_model.add(Dropout(0.3))

lstm_model.add(LSTM(256))
lstm_model.add(Dropout(0.3))

lstm_model.add(Dense(vocab_size, activation='softmax'))

In [74]:
lstm_model.compile(
    optimizer = 'adam',
    loss = 'categorical_crossentropy',
    metrics = ['accuracy']
)

In [75]:
lstm_model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [76]:
epochs = 100
batch_size = 128

In [41]:
histoy_lstm = lstm_model.fit(
    X_padded, y_one_hot,
    epochs = epochs,
    batch_size = batch_size,
    validation_split = 0.1
)

Epoch 1/100
 43/600 ━━━━━━━━━━━━━━━━━━━━ 2:17:54 15s/step - accuracy: 0.0361 - loss: 8.3208

KeyboardInterrupt: 

In [ ]:
lstm_model.save("lstm_model.h5")


In [77]:
from tensorflow.keras.models import load_model
lstm_model = load_model("lstm_model (1) (1).h5")

In [78]:
index_to_word = {index: word for word, index in tokenizer.word_index.items()}
index_to_word[0] = "<PAD>"   # optional but useful

In [79]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [80]:
def sample(preds, temperature=0.8):
    preds = np.asarray(preds).astype('float64')
    preds = np.log(preds + 1e-8) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    return np.random.choice(len(preds), p=preds)

In [81]:
def predictor(model, tokenizer, text, max_len, index_to_word):
    seq = tokenizer.texts_to_sequences([text])[0]
    seq = pad_sequences([seq], maxlen=max_len, padding='pre')

    pred = model.predict(seq, verbose=0)
    pred_index = sample(pred[0], temperature=0.8)

    # Safe lookup to avoid KeyError
    return index_to_word.get(pred_index, "<UNK>")

In [82]:
seed_text = "what are you"
next_word = predictor(lstm_model, tokenizer, seed_text, max_len, index_to_word)
print(next_word)

am


In [83]:
def generate_text(model, tokenizer, seed_text, max_len, num_words, index_to_word):
    result = seed_text

    for _ in range(num_words):
        seq = tokenizer.texts_to_sequences([result])[0]
        seq = pad_sequences([seq], maxlen=max_len, padding='pre')

        pred = model.predict(seq, verbose=0)
        pred_index = sample(pred[0], 0.8)

        word = index_to_word.get(pred_index, "")

        if word == "" or word == result.split()[-1]:
            continue

        result += " " + word

    return result

In [84]:
output = generate_text(lstm_model, tokenizer, seed_text, max_len, 10, index_to_word)
print(output)

what are you made <OOV> die as one money have and they you


In [88]:
import pickle
with open("tokenizer.pkl","wb") as f:
  pickle.dump(tokenizer, f)

In [89]:
with open("max_len.pkl", "wb") as f:
  pickle.dump(max_len, f)

In [87]:
seed_text = "Life is a"
output = generate_text(lstm_model, tokenizer, seed_text, max_len, 10, index_to_word)
print(output)

Life is a from going like a them somewhere up worthy a stars
